In [15]:
# Importing necessary libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import random
import pandas as pd

In [16]:
options = Options()
options.add_argument("--start-maximized")

In [17]:
driver = webdriver.Chrome(service=Service(), options = options)

### Batch-wise Web Scraping from Hotfrog (Australia)

This code scrapes motorcycle spare parts business listings from **Hotfrog Australia** across multiple pages.
- Divides the scraping into **batches**, each covering 10 pages of search results
- For each business entry, it extracts:
  - `Company Name`
  - `Phone Number`
  - `Location / Address`
- Uses Selenium to load the page and extract elements dynamically
- Stores each batch of results in a separate CSV file (`batch_1.csv`, `batch_2.csv`, etc.)

#### Key Features:
- Uses `try-except` blocks to handle missing data safely
- Adds a **random delay** between batches to avoid getting blocked
- Total of 14 batches = 140 pages scraped
- Final message confirms completion


In [21]:
for batch_num in range(1, 15): 
    company_names, phones, locations = [], [], []

    # Start and end page numbers for this batch
    start_page = (batch_num - 1) * 10 + 1
    end_page = start_page + 9

    for page in range(start_page, end_page + 1):
        url = f"https://www.hotfrog.com.au/search/australia/motorcycle%20spare%20parts/{page}"
        driver.get(url)
        time.sleep(3)

        result_boxes = driver.find_elements(By.CLASS_NAME, "hf-box")

        for box in result_boxes:
            try:
                name = box.find_element(By.TAG_NAME, "h3").text.strip()
            except:
                name = "N/A"
            try:
                phone = box.find_element(By.CSS_SELECTOR, "a[href^='tel']").text.strip()
            except:
                phone = "N/A"
            try:
                span_tags = box.find_elements(By.CLASS_NAME, "small")
                address = next((s.text.strip() for s in span_tags if ',' in s.text), "N/A")
            except:
                address = "N/A"
            company_names.append(name)
            phones.append(phone)
            locations.append(address)

        print(f"Scraped page {page}")

    df = pd.DataFrame({
        "Company Name": company_names,
        "Phone": phones,
        "Location": locations
    })
    print(f"Batch {batch_num}: {len(company_names)} entries collected.")
    print(f"Batch {batch_num}: {len(phones)} entries collected.")
    print(f"Batch {batch_num}: {len(locations)} entries collected.")

    df.to_csv(f"batch_{batch_num}.csv", index=False)
    print(f"Saved batch {batch_num} as batch_{batch_num}.csv")

    time.sleep(random.randint(10, 20))  # random wait before next batch

driver.quit()
print("All batches scraped and saved.")

Scraped page 1
Scraped page 2
Scraped page 3
Scraped page 4
Scraped page 5
Scraped page 6
Scraped page 7
Scraped page 8
Scraped page 9
Scraped page 10
Batch 1: 120 entries collected.
Batch 1: 120 entries collected.
Batch 1: 120 entries collected.
Saved batch 1 as batch_1.csv
Scraped page 11
Scraped page 12
Scraped page 13
Scraped page 14
Scraped page 15
Scraped page 16
Scraped page 17
Scraped page 18
Scraped page 19
Scraped page 20
Batch 2: 120 entries collected.
Batch 2: 120 entries collected.
Batch 2: 120 entries collected.
Saved batch 2 as batch_2.csv
Scraped page 21
Scraped page 22
Scraped page 23
Scraped page 24
Scraped page 25
Scraped page 26
Scraped page 27
Scraped page 28
Scraped page 29
Scraped page 30
Batch 3: 120 entries collected.
Batch 3: 120 entries collected.
Batch 3: 120 entries collected.
Saved batch 3 as batch_3.csv
Scraped page 31
Scraped page 32
Scraped page 33
Scraped page 34
Scraped page 35
Scraped page 36
Scraped page 37
Scraped page 38
Scraped page 39
Scraped pa

In [ ]:
### Merging All Batches into a Single Dataset

After scraping each batch of Hotfrog data into separate CSV files (`batch_1.csv` to `batch_14.csv`), this cell combines all the batches into one comprehensive DataFrame.

####  What This Code Does:
- Reads all individual batch CSVs into separate DataFrames
- Concatenates them using `pd.concat()` with `ignore_index=True` to reset row indices
- Saves the combined data into a single CSV file named `combined_scraped_data.csv`
- Prints the total number of entries to confirm successful merge

> This step is essential to unify all scraped records into a single file for easier cleaning, email enrichment, and final analysis.


In [22]:
import pandas as pd
files = ["batch_1.csv", "batch_2.csv", "batch_3.csv", "batch_4.csv", "batch_5.csv", "batch_6.csv", "batch_7.csv", "batch_8.csv", "batch_9.csv", "batch_10.csv", "batch_11.csv", "batch_12.csv", "batch_13.csv", "batch_14.csv"]  # Add as many as you have

dfs = [pd.read_csv(file) for file in files]
combined_df = pd.concat(dfs, ignore_index=True)

print("Combined entries:", len(combined_df))
combined_df.to_csv("combined_scraped_data.csv", index=False)
print("Combined CSV saved successfully!")

Combined entries: 1680
Combined CSV saved successfully!


### Email Enrichment via Hunter.io and Clearbit API

This cell attempts to enrich the scraped company data with emails using:

- **Clearbit API**: To fetch company domains from names
- **Hunter.io API**: To get emails from those domains

Each company name was processed, and a short delay was added to respect rate limits.

> Due to Hunter.io's limited free credits and the fact that most small businesses don’t have public domains or emails, only a small portion of entries were successfully enriched. The rest are marked as `None`.


In [24]:
import requests
import pandas as pd
import time

API_KEY = "527077e90034be2da4ea1d42e57571f5e6c1664b"


In [25]:
len(company_names)

120

In [26]:
def get_domain_from_clearbit(company_name):
    try:
        response = requests.get(f"https://autocomplete.clearbit.com/v1/companies/suggest?query={company_name}")
        data = response.json()
        if data and len(data) > 0:
            return data[0]['domain']
        return None
    except:
        return None


In [27]:
def get_email_from_hunter(domain):
    try:
        response = requests.get(f"https://api.hunter.io/v2/domain-search?domain={domain}&api_key={API_KEY}")
        data = response.json()
        if 'data' in data and 'emails' in data['data'] and len(data['data']['emails']) > 0:
            return data['data']['emails'][0]['value']
        return None
    except:
        return None


In [28]:
df = pd.read_csv("combined_scraped_data.csv") 
emails = []

for company in df['Company Name']:
    print(f"Processing: {company}")
    domain = get_domain_from_clearbit(company)
    if domain:
        print(f" → Domain found: {domain}")
        email = get_email_from_hunter(domain)
        print(f" → Email found: {email}")
    else:
        print(" → No domain found.")
        email = None
    emails.append(email)
    time.sleep(1.5)  # Sleep to avoid being blocked




Processing: First Class Motorcycles Yamaha Spare Parts
 → No domain found.
Processing: City Motorcycle Spares
 → No domain found.
Processing: Redfern Motorcycle Spares
 → No domain found.
Processing: British Motorcycle Spares
 → No domain found.
Processing: Queensland Motorcycle Spares
 → No domain found.
Processing: Recycle Motorcycle Spares
 → No domain found.
Processing: Metropolitan Motorcycle Spares Nsw
 → No domain found.
Processing: Browns Plains Motorcycle Spares
 → No domain found.
Processing: Motorcycle Parts
 → Domain found: motorcyclepartswarehouse.co.uk
 → Email found: support@motorcyclepartswarehouse.co.uk
Processing: Gladesville Auto Spare Parts
 → No domain found.
Processing: Gaitex Auto Spare Parts
 → No domain found.
Processing: Elite Spare Parts
 → Domain found: elitespareparts.com
 → Email found: None
Processing: Saga Motorcycle Parts
 → No domain found.
Processing: Pro Motorcycle Parts
 → No domain found.
Processing: Collie Motorcycle Parts
 → No domain found.
Proc

KeyboardInterrupt: 

In [34]:
while len(emails) < len(df):
    emails.append(None)
df['Email'] = emails
df.to_csv("25-0020-I.csv", index=False)

In [50]:
df = pd.read_csv("25-0020-I.csv")

# Drop duplicates by company name
df.drop_duplicates(subset='Company Name', inplace=True)

# Drop rows where company name and phone number are missing
df.dropna(subset=['Company Name', 'Phone'], inplace=True)

###  Data Cleaning and Preprocessing

This cell performs final preprocessing on the scraped dataset to ensure data quality:

- Removes duplicate companies based on the `Company Name`
- Drops entries missing both `Company Name` and `Phone`
- Validates email format using a regular expression
- Resets the DataFrame index after cleaning

> After preprocessing, the total number of rows reduced from **1680 to 1411**, ensuring that only clean and meaningful records remain for analysis or submission.


In [52]:
# Basic email format validation
import re
def is_valid_email(email):
    if not isinstance(email, str): return False
    return bool(re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['Email'] = df['Email'].where(df['Email'].apply(is_valid_email))

# Reset index
df.reset_index(drop=True, inplace=True)
df.to_csv("Australia.csv", index=False)
